# 10 — Temporal CLV Target (Leakage Fix)
## Customer Analytics Platform

Purpose: Replace the same-snapshot RFM-threshold target from 05/06
with a genuinely future target. Features are computed only from
orders before a cutoff date. The target is computed only from
orders after that cutoff. No column used as a feature is derived
from the same time window as the label.

Target: is_repeat_customer_90d
1 = customer placed at least one order in the 90 days after the cutoff
0 = customer did not

In [0]:
# install xgboost - each notebook has its own python environment
# installing it in 09 doesn't carry over here

%pip install xgboost

In [0]:
dbutils.library.restartPython()

In [0]:
# load bronze customers (NOT silver) - silver's dedup step drops customer_id
# rows that orders_df still references, which causes broken joins below

bronze_path = "/Volumes/workspace/default/olist_raw_data/bronze"
silver_path = "/Volumes/workspace/default/olist_raw_data/silver"

customers_bronze_df = spark.read.format("delta").load(f"{bronze_path}/customers")
orders_df           = spark.read.format("delta").load(f"{silver_path}/orders")
payments_df         = spark.read.format("delta").load(f"{silver_path}/payments")

print("tables loaded")

In [0]:
# check the actual date range before picking a cutoff

from pyspark.sql.functions import min as spark_min, max as spark_max

orders_df.select(
    spark_min("order_purchase_timestamp").alias("earliest_order"),
    spark_max("order_purchase_timestamp").alias("latest_order")
).show(truncate=False)

In [0]:
# join orders with payments and customer_unique_id once, reuse for both windows
# using bronze customers here, plus a safety filter to drop any remaining nulls

from pyspark.sql.functions import col

orders_payments_customers = orders_df \
    .join(payments_df, "order_id", "left") \
    .join(customers_bronze_df.select("customer_id", "customer_unique_id"), "customer_id", "left") \
    .select(
        "customer_unique_id",
        "order_id",
        "order_purchase_timestamp",
        "total_payment_value"
    ) \
    .filter(col("customer_unique_id").isNotNull())

print(f"orders joined : {orders_payments_customers.count():,}")

In [0]:
# confirm the join fix worked - should be close to 0 now

null_customer_count = orders_df \
    .join(payments_df, "order_id", "left") \
    .join(customers_bronze_df.select("customer_id", "customer_unique_id"), "customer_id", "left") \
    .filter(col("customer_unique_id").isNull()) \
    .count()

print(f"rows with null customer_unique_id (bronze join) : {null_customer_count:,}")

In [0]:
# define the cutoff and the future label window

feature_cutoff_date = "2018-01-01"
#label_window_end    = "2018-04-01"   # cutoff + 90 days
label_window_end    = "2018-07-01"   # cutoff + 180 days

print(f"feature window : orders before {feature_cutoff_date}")
print(f"label window   : orders from {feature_cutoff_date} to {label_window_end}")

In [0]:
# historical orders only - used for recency, frequency, monetary

from pyspark.sql.functions import (
    max as spark_max, count, sum as spark_sum, round as spark_round,
    datediff, lit, to_date
)

historical_orders = orders_payments_customers.filter(
    col("order_purchase_timestamp") < feature_cutoff_date
)

rfm_historical = historical_orders \
    .groupBy("customer_unique_id") \
    .agg(
        datediff(
            to_date(lit(feature_cutoff_date)),
            spark_max("order_purchase_timestamp")
        ).alias("recency"),
        count("order_id").alias("frequency"),
        spark_round(spark_sum("total_payment_value"), 2).alias("monetary")
    ) \
    .fillna({"monetary": 0.0})

print(f"customers with history before cutoff : {rfm_historical.count():,}")
rfm_historical.show(5)

In [0]:
# future orders only - used to build the target

future_orders = orders_payments_customers.filter(
    (col("order_purchase_timestamp") >= feature_cutoff_date) &
    (col("order_purchase_timestamp") < label_window_end)
)

future_activity = future_orders \
    .groupBy("customer_unique_id") \
    .agg(
        count("order_id").alias("future_order_count"),
        spark_round(spark_sum("total_payment_value"), 2).alias("future_monetary")
    )

print(f"customers with future activity : {future_activity.count():,}")
future_activity.show(5)

In [0]:
# join historical features with future activity

modeling_df = rfm_historical \
    .join(future_activity, "customer_unique_id", "left") \
    .fillna({"future_order_count": 0, "future_monetary": 0.0}) \
    .withColumn(
        "is_repeat_customer_90d",
        (col("future_order_count") > 0).cast("int")
    )

print(f"final modeling set : {modeling_df.count():,} customers")
print("\ntarget distribution:")
modeling_df.groupBy("is_repeat_customer_90d").count().show()

In [0]:
# overlap check - direct intersection, independent of the join above

overlap_check = rfm_historical.select("customer_unique_id") \
    .intersect(future_activity.select("customer_unique_id")) \
    .count()

print(f"customers appearing in both historical and future sets : {overlap_check}")

In [0]:
# class balance as a percentage

from pyspark.sql.functions import count as spark_count

total = modeling_df.count()
modeling_df.groupBy("is_repeat_customer_90d") \
    .agg(spark_count("*").alias("customers")) \
    .withColumn("pct", spark_round(col("customers") / total * 100, 2)) \
    .show()

In [0]:
# save as a new gold table

gold_path = "/Volumes/workspace/default/olist_raw_data/gold"

modeling_df.write.format("delta") \
    .mode("overwrite") \
    .save(f"{gold_path}/rfm_temporal_features")

print("temporal features saved")

In [0]:
# convert to pandas for xgboost

import pandas as pd

pdf = modeling_df.select(
    "recency", "frequency", "monetary", "is_repeat_customer_90d"
).toPandas()

pdf = pdf.fillna(0)

print(f"dataset shape : {pdf.shape}")
print(pdf["is_repeat_customer_90d"].value_counts())

In [0]:
# train/test split

from sklearn.model_selection import train_test_split

X = pdf[["recency", "frequency", "monetary"]]
y = pdf["is_repeat_customer_90d"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print(f"train : {X_train.shape[0]} rows")
print(f"test  : {X_test.shape[0]} rows")
print(f"scale_pos_weight : {scale_pos_weight:.2f}")

In [0]:
# pull the tuned hyperparameters from the hyperopt run in 09

import mlflow

mlflow.set_experiment(
    "/Users/elitahazelgorimanikonda@gmail.com/CLV_Customer_Segmentation"
)

runs_v3 = mlflow.search_runs(
    filter_string="tags.mlflow.runName = 'xgboost_clv_v3_tuned'",
    order_by=["metrics.roc_auc DESC"]
)

if not runs_v3.empty:
    tuned_params = {
        "max_depth":        int(runs_v3.iloc[0]["params.max_depth"]),
        "learning_rate":    float(runs_v3.iloc[0]["params.learning_rate"]),
        "n_estimators":     int(runs_v3.iloc[0]["params.n_estimators"]),
        "subsample":        float(runs_v3.iloc[0]["params.subsample"]),
        "colsample_bytree": float(runs_v3.iloc[0]["params.colsample_bytree"]),
        "min_child_weight": int(runs_v3.iloc[0]["params.min_child_weight"]),
        "gamma":            float(runs_v3.iloc[0]["params.gamma"])
    }
    print("using tuned params from xgboost_clv_v3_tuned")
else:
    tuned_params = {
        "max_depth": 4, "learning_rate": 0.1, "n_estimators": 100,
        "subsample": 0.9, "colsample_bytree": 0.9,
        "min_child_weight": 3, "gamma": 0.5
    }
    print("v3 tuned run not found - using fallback defaults")

print(tuned_params)

In [0]:
# train v4 on the temporal target, log to mlflow

import xgboost as xgb
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score
)

with mlflow.start_run(run_name="xgboost_clv_v4_temporal_target"):

    model_v4 = xgb.XGBClassifier(
        **tuned_params,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric="logloss",
        verbosity=0
    )
    model_v4.fit(X_train, y_train)

    y_pred      = model_v4.predict(X_test)
    y_pred_prob = model_v4.predict_proba(X_test)[:, 1]

    accuracy   = accuracy_score(y_test, y_pred)
    precision  = precision_score(y_test, y_pred)
    recall     = recall_score(y_test, y_pred)
    f1         = f1_score(y_test, y_pred)
    roc_auc    = roc_auc_score(y_test, y_pred_prob)
    avg_prec   = average_precision_score(y_test, y_pred_prob)

    mlflow.log_params(tuned_params)
    mlflow.log_param("scale_pos_weight", scale_pos_weight)
    mlflow.log_param("target", "is_repeat_customer_90d")
    mlflow.log_param("feature_cutoff_date", feature_cutoff_date)
    mlflow.log_param("label_window_end", label_window_end)

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("average_precision", avg_prec)

    mlflow.xgboost.log_model(model_v4, "xgboost_clv_v4_temporal")

    print("V4 Temporal Target Model Complete")
    print("-" * 40)
    print(f"accuracy           : {accuracy:.4f}")
    print(f"precision           : {precision:.4f}")
    print(f"recall              : {recall:.4f}")
    print(f"f1 score            : {f1:.4f}")
    print(f"roc auc             : {roc_auc:.4f}")
    print(f"average precision   : {avg_prec:.4f}")
    print("-" * 40)
    print("v4 logged to mlflow")

In [0]:
# feature importance

import matplotlib.pyplot as plt

feature_names = ["recency", "frequency", "monetary"]
importance    = model_v4.feature_importances_

plt.figure(figsize=(8, 4))
plt.barh(feature_names, importance)
plt.xlabel("importance score")
plt.title("XGBoost Feature Importance — V4 Temporal Target")
plt.tight_layout()
plt.show()

for name, score in zip(feature_names, importance):
    print(f"  {name:<12} : {score:.4f}")

## Temporal CLV Target Summary

Target: is_repeat_customer_90d, computed strictly after a cutoff date,
using customer_unique_id sourced from bronze customers (silver's dedup
step was dropping valid customer_id references and creating null-key
join corruption in earlier attempts).

Model: XGBoost V4, tuned hyperparameters from 09_Hyperparameter_Tuning,
scale_pos_weight applied for class imbalance.

Window comparison tested: 90 days vs 180 days.
- 90-day:  ROC AUC 0.5293, AP 0.0206 (2.86x lift over 0.72% base rate)
- 180-day: ROC AUC 0.5461, AP 0.0286 (2.23x lift over 1.28% base rate)

Conclusion: widening the window did not meaningfully improve
discrimination. Recency/frequency/monetary alone carry weak signal
for predicting future repeat purchase - this is a genuine finding,
not a tuning failure. The features that made V1/V2 look artificially
strong (because they WERE the label) are much weaker once measured
honestly against true future behavior.

Next iteration would need richer features: review sentiment/score,
product category, delivery experience, time-of-year seasonality -
none of which existed in the leaky RFM-only formulation.

Next Step:
- This finding closes out the leakage-fix enhancement - the target
  is now genuinely non-leaky and honestly measured, even though
  performance is weak
- Real-time serving and drift monitoring (next roadmap items) should
  still wrap this V4 model, not V2/V3, despite the weak metrics -
  it's the only one that reflects true predictive difficulty
- A future richer-feature version (review score, category, delivery
  experience) is the right next iteration if CLV prediction quality
  needs to improve, not more tuning on the current 3 features